### Install required package, download required files and import base libraries

In [1]:
#install cobrapy package
!pip install cobra --quiet


#download the custom library
!wget -q https://raw.githubusercontent.com/sshameer/MultiphaseFBAProtocol/refs/heads/main/studyFunctions.py
#downloading the model
!wget -q https://raw.githubusercontent.com/sshameer/MultiphaseFBAProtocol/refs/heads/main/PlantCoreMetabolism_v2_0_0.xml

#adding the fractional charges of the metabolites as a csv file
!wget -q https://raw.githubusercontent.com/sshameer/MultiphaseFBAProtocol/refs/heads/main/FractionalCharges.csv

#Adding the organic solute concentratation data (Shameer et al., 2020)
!wget -q https://raw.githubusercontent.com/sshameer/MultiphaseFBAProtocol/refs/heads/main/ProcessedData_OrganicSolutes_Starch.csv


#import library
import cobra
from cobra import io, flux_analysis, util
from cobra.io import read_sbml_model
from cobra.core import Reaction, Metabolite
from IPython import display
#from studyFunctions import setupMultiphaseModel
import time
#import the custom library
from studyFunctions import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 19.0 MB/s eta 0:00:00


# Part I: Build base model



#### Step 1A: Create multiple copies of the stoichiometric model

In [2]:
model = read_sbml_model("PlantCoreMetabolism_v2_0_0.xml")

In [3]:
#Create multiple copies of the stoichiometric model
modelDict = dict()

for i in range(1,11):
  temp = model.copy()

  for met in temp.metabolites:
    met.id = met.id+str(i)
    met.compartment = met.compartment+str(i)
  for rxn in temp.reactions:
    rxn.id = rxn.id+str(i)

  modelDict[i] = temp

for i in range(1,11):
  if i  == 1:
    FruitModel = modelDict[i].copy()
  else:
    FruitModel.merge(modelDict[i])

    ## see Note 3
    for met in modelDict[i].metabolites:
      if not FruitModel.metabolites.has_id(met.id):
        FruitModel.add_metabolites([met.copy()])






In [4]:
#example for reactions copied
FruitModel.reactions.get_by_id("GLUCURONOKINASE_RXN_c1").reaction


'0.65 ATP_c1 + GLUCURONATE_c1 + 0.35 aATP_c1 --> 0.5 ADP_c1 + CPD_510_c1 + 0.85 PROTON_c1 + 0.5 aADP_c1'

In [5]:
#example for reactions copied
FruitModel.reactions.get_by_id("GLUCURONOKINASE_RXN_c2").reaction

'0.65 ATP_c2 + GLUCURONATE_c2 + 0.35 aATP_c2 --> 0.5 ADP_c2 + CPD_510_c2 + 0.85 PROTON_c2 + 0.5 aADP_c2'

In [6]:
cobra_model = FruitModel.copy()

#### Step 1B :Constrain each temporal phase for a pericarp sink tissue



In [7]:
#remove free sucrose, glucose, NH4 and light uptake making it sink specific
for i in range(1,11):
    cobra_model.reactions.get_by_id("Sucrose_tx"+str(i)).lower_bound=0
    cobra_model.reactions.get_by_id("Sucrose_tx"+str(i)).upper_bound=0
    cobra_model.reactions.get_by_id("GLC_tx"+str(i)).lower_bound=0
    cobra_model.reactions.get_by_id("GLC_tx"+str(i)).upper_bound=0
    cobra_model.reactions.get_by_id("NH4_tx"+str(i)).lower_bound=0
    cobra_model.reactions.get_by_id("NH4_tx"+str(i)).upper_bound=0
    cobra_model.reactions.get_by_id("Photon_tx"+str(i)).lower_bound=0
    cobra_model.reactions.get_by_id("Photon_tx"+str(i)).upper_bound=0


In [8]:
#similarly phloem contribution is also updated to phloem uptake to making it sink specific
#for example sucrose metabolite is added to this reaction representing its transfer from boundary (phloem) into the system (cytosol)
for i in range(1,11):
    rxn = cobra_model.reactions.get_by_id("Phloem_output_tx"+str(i))
    #print rxn.reaction
    metlist = rxn.metabolites.keys()
    for met in metlist:
        if met.id.__contains__("PROTON"):
            continue
        a = rxn.metabolites.get(met)
        rxn.add_metabolites({met:-2*a})
    c = rxn.metabolites.get(cobra_model.metabolites.get_by_id("sSUCROSE_b"+str(i)))
    rxn.add_metabolites({cobra_model.metabolites.get_by_id("sSUCROSE_b"+str(i)):-1*c})
    rxn.add_metabolites({cobra_model.metabolites.get_by_id("SUCROSE_c"+str(i)):c})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    #print rxn.reaction
    #break
    rxn.id = "Phloem_uptake_tx"+str(i)

cobra_model.reactions._generate_index()

##### Remove or constraint unwanted reactions to 0

In [9]:
#removed reactions not related significant in the reaction
for i in range(1,10):
    cobra_model.reactions.get_by_id("Ca_biomass"+str(i)).remove_from_model()
    cobra_model.reactions.get_by_id("Mg_biomass"+str(i)).remove_from_model()
    cobra_model.reactions.get_by_id("K_biomass"+str(i)).remove_from_model()


for i in range(1,11):
    rxn=cobra_model.reactions.get_by_id("AraCore_Biomass_tx"+str(i))
    rxn.lower_bound=0
    rxn.upper_bound=0

for i in range(1,11):
    r=cobra_model.reactions.get_by_id("Biomass_tx"+str(i))
    r.upper_bound = 0
    r.lower_bound = 0

#### Step 1C: Set up linker reactions

In [10]:
#setting up "linker reactions" or "psuedoreactions" connecting one temporal phase to another
#for example using starch as the linker metabolite, psuedoreactions - transfering the starch from one temporal phase to its counter part in next temporal phase is created
#cytosolic and plastidic transfer reactions
cobra_model2 = cobra_model.copy()
for i in range(1,10):
    k = "STARCH"
    rxn = Reaction(k+"_p_Transfer"+str(i)+str(i+1))
    rxn.name = k+"_p_Transfer"+str(i)+str(i+1)
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_p"+str(i)):-1,cobra_model2.metabolites.get_by_id(k+"_p"+str(i+1)):1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    rxn.notes["influx_stage"]=i+1
    rxn.notes["outflux_stage"]=i
    cobra_model2.add_reactions([rxn])

    for k in ["SUCROSE","GLC","FRU","MAL","CIT","FUM"]:
        rxn = Reaction(k+"_c_Transfer"+str(i)+str(i+1))
        rxn.name = k+"_Transfer"+str(i)+str(i+1)
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_c"+str(i)):-1,cobra_model2.metabolites.get_by_id(k+"_c"+str(i+1)):1})
        rxn.lower_bound = 0
        rxn.upper_bound = 1000
        rxn.notes["influx_stage"]=i+1
        rxn.notes["outflux_stage"]=i
        cobra_model2.add_reactions([rxn])

    for k in ["GLN","ASN","SER","GLY","THR","L_ALPHA_ALANINE","4_AMINO_BUTYRATE","VAL","ILE","PHE","LEU","LYS","ARG","L_ASPARTATE","GLT","HIS","MET","PRO","TRP","TYR","CYS"]:
        rxn = Reaction(k+"_c_Transfer"+str(i)+str(i+1))
        rxn.name = k+"_Transfer"+str(i)+str(i+1)
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_c"+str(i)):-1,cobra_model2.metabolites.get_by_id(k+"_c"+str(i+1)):1})
        rxn.lower_bound = 0
        rxn.upper_bound = 1000
        rxn.notes["influx_stage"]=i+1
        rxn.notes["outflux_stage"]=i
        cobra_model2.add_reactions([rxn])


#vacuolar transfer reactions
import re
fin = open("FractionalCharges.csv","r")

ChargeDict=dict()
for line in fin:
  met=line.replace("\n","").split("\t")[0]
  met = met.replace("-","_")
  charge = line.replace("\n","").split("\t")[1]
  ChargeDict[met]=charge

fin.close()

for met in cobra_model2.metabolites:
  tempMet=met.id
  if(met.id[len(met.id)-1]=="2" or met.id[len(met.id)-1]=="1"):
    tempMet = met.id[0:len(met.id)-1]
  if(ChargeDict.keys().__contains__(tempMet)):
    met.charge = ChargeDict.get(tempMet)
  if met.charge is None:
    met.charge=0


from cobra.core import Metabolite, Reaction

#Adding transfer reactions
###################
#tmset =set()
#for met in cobra_model2.metabolites:
#    if met.compartment.__contains__("v") and not met.compartment == "v10":
#        tmset.add(met.id[0:len(met.id)-1])

vacMets=["SUCROSE_v","MAL_v","AMMONIUM_v","CIT_v","GLN_v","ASN_v","SER_v","GLN_v","GLY_v","THR_v","L_ALPHA_ALANINE_v","4_AMINO_BUTYRATE_v","VAL_v","ILE_v","PHE_v","LEU_v","LYS_v","ARG_v","L_ASPARTATE_v","GLT_v","bHIS_v","MET_v","PRO_v","TRP_v","TYR_v","CYS_v","GLC_v","FRU_v","FUM_v","MGII_v","KI_v","CAII_v","NITRATE_v"]
tmset = set(vacMets)

for i in range(1,10):
    for met in tmset:
        tempRxn = Reaction(met+"_Transfer"+str(i)+str(i+1))
        tempRxn.add_metabolites({cobra_model2.metabolites.get_by_id(met+str(i)):-1,cobra_model2.metabolites.get_by_id(met+str(i+1)):1})
        tempRxn.lower_bound=0
        tempRxn.upper_bound=1000
        tempRxn.notes["influx_stage"]=i+1
        tempRxn.notes["outflux_stage"]=i
        cobra_model2.add_reactions([tempRxn])


fractionMets=dict()
for rxn in cobra_model2.reactions:
    for met in rxn.metabolites.keys():
        a=re.search("^a{1,3}",met.id)
        anion=""
        if a:
            anion=a.group(0)
        b=re.search("^b{1,3}",met.id)
        basic=""
        if b:
            basic=b.group(0)
        prefix = anion
        if prefix == "":
            prefix = basic
        if (abs(rxn.metabolites.get(met)) % 1 > 0 and (not prefix == "") and met.compartment == "v1"):
            fractionMets[met]=prefix

temp=cobra_model2.copy()
for i in range(1,10):
    for met in fractionMets.keys():
        for rxn in met.reactions:
            if rxn.id.__contains__("_Transfer"):
                continue
            else:
                mainMet = met.id[len(fractionMets[met]):]
                coeff1 = temp.reactions.get_by_id(rxn.id).metabolites.get(temp.metabolites.get_by_id(mainMet))
                coeff2 = temp.reactions.get_by_id(rxn.id).metabolites.get(temp.metabolites.get_by_id(met.id))
                total = coeff1 + coeff2
                coeff1 = float(coeff1)/total
                coeff2 = float(coeff2)/total
                if cobra_model2.reactions.has_id(mainMet[0:len(mainMet)-1]+"_Transfer"+str(i)+str(i+1)):
                    if cobra_model2.reactions.has_id(met.id[0:len(met.id)-1]+"_Transfer"+str(i)+str(i+1)):
                        temp.reactions.get_by_id(met.id[0:len(met.id)-1]+"_Transfer"+str(i)+str(i+1)).remove_from_model()
                    temp.reactions.get_by_id(mainMet[0:len(mainMet)-1]+"_Transfer"+str(i)+str(i+1)).remove_from_model()
                    Reac = Reaction(mainMet[0:len(mainMet)-1]+"_Transfer"+str(i)+str(i+1),name=mainMet+"_Transfer"+str(i)+str(i+1))
                    Reac.add_metabolites({temp.metabolites.get_by_id(met.id[0:len(met.id)-1]+str(i)):-coeff2,temp.metabolites.get_by_id(met.id[0:len(met.id)-1]+str(i+1)):coeff2,temp.metabolites.get_by_id(mainMet[0:len(mainMet)-1]+str(i)):-coeff1,temp.metabolites.get_by_id(mainMet[0:len(mainMet)-1]+str(i+1)):coeff1})
                    Reac.upper_bound=1000
                    temp.add_reactions([Reac])
                    #print Reac.reaction
                    break

#####################


cobra_model2 = temp.copy()
del(temp)                  #to help clear memory


# Part II: Configure the initial and final states


Import experimental data collected from [Shameer et. al., 2020](https://github.com/ljs1002/Shameer-et-al-Predicting-metabolism-during-growth-by-osmotic-cell-expansion/tree/master)

In [11]:
import pandas

df = pandas.read_csv("/content/ProcessedData_OrganicSolutes_Starch.csv",delimiter="\t")

DPA = list(df.DPA)
Conc=dict()

Conc["MAL"]=list(df.MAL)
Conc["CIT"]=list(df.CIT)
Conc["FUM"]=list(df.FUM)
Conc["SUCROSE"]=list(df.SUC)
Conc["FRU"]=list(df.FRU)
Conc["GLC"]=list(df.GLC)
Conc["L_ALPHA_ALANINE"]=list(df.L_ALPHA_ALANINE)
Conc["ASN"]=list(df.ASN)
Conc["L_ASPARTATE"]=list(df.L_ASPARTATE)
Conc["4_AMINO_BUTYRATE"]=list(df.GABA)
Conc["GLT"]=list(df.GLT)
Conc["GLN"]=list(df.GLN)
Conc["GLY"]=list(df.GLY)
Conc["ILE"]=list(df.ILE)
Conc["LYS"]=list(df.LYS)
Conc["MET"]=list(df.MET)
Conc["PHE"]=list(df.PHE)
Conc["PRO"]=list(df.PRO)
Conc["SER"]=list(df.SER)
Conc["THR"]=list(df.THR)
Conc["TRP"]=list(df.TRP)
Conc["TYR"]=list(df.TYR)
Conc["VAL"]=list(df.VAL)
Conc["STARCH"]=list(df.STARCH)


AA = ["GLN_c","ASN_c","SER_c","GLY_c","THR_c","L_ALPHA_ALANINE_c","4_AMINO_BUTYRATE_c","VAL_c","ILE_c","PHE_c","LEU_c","LYS_c","ARG_c","L_ASPARTATE_c","GLT_c","HIS_c","MET_c","PRO_c","TRP_c","TYR_c","CYS_c"]


#### Step 20: Setting up biomass reactions for core the growth requirements

In [12]:
#biomass requirement for protein, nucleic acid, lipid represented as seperate biomass reactions
#example biomass reaction for DNA, with GC content = 60%
for i in range(1,11):
    rxn = Reaction("Biomass_Lipid_tx"+str(i))
    rxn.name = "Biomass_Lipid"
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("PHOSPHATIDYL_CHOLINE_r"+str(i)):-0.353,cobra_model2.metabolites.get_by_id("L_1_PHOSPHATIDYL_ETHANOLAMINE_r"+str(i)):-0.374,cobra_model2.metabolites.get_by_id("L_PHOSPHATIDATE_p"+str(i)):-0.273})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])

    rxn = cobra_model2.reactions.get_by_id("Biomass_tx"+str(i))
    rxn.name = "Biomass_Protein"
    met = Metabolite("PROTEIN_b"+str(i))
    met.name="PROTEIN_b"+str(i)
    met.compartment = "b"+str(i)
    rxn.add_metabolites({met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000

    #remove cations from Biomass_tx
    cobra_model2.metabolites.get_by_id("K_b"+str(i)).remove_from_model()
    cobra_model2.metabolites.get_by_id("Ca_b"+str(i)).remove_from_model()
    cobra_model2.metabolites.get_by_id("Mg_b"+str(i)).remove_from_model()

    #create biomass constraints for DNA and RNA with GC content = 60%
    GC = 0.6

    rxn = Reaction("Biomass_DNA_tx"+str(i))
    rxn.name="Biomass_DNA"
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("ATP_c"+str(i)):-(1-GC)*0.65,cobra_model2.metabolites.get_by_id("aATP_c"+str(i)):-(1-GC)*0.35,cobra_model2.metabolites.get_by_id("GTP_c"+str(i)):-GC*0.65,cobra_model2.metabolites.get_by_id("aGTP_c"+str(i)):-GC*0.35,cobra_model2.metabolites.get_by_id("UTP_p"+str(i)):-(1-GC)*0.18,cobra_model2.metabolites.get_by_id("aUTP_p"+str(i)):-(1-GC)*0.82,cobra_model2.metabolites.get_by_id("CTP_p"+str(i)):-GC*0.79,cobra_model2.metabolites.get_by_id("aCTP_p"+str(i)):-GC*0.21})
    met = Metabolite("DNA_b"+str(i))
    met.name="DNA_b"+str(i)
    met.compartment = "b"+str(i)
    rxn.add_metabolites({met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])

    rxn = Reaction("Biomass_RNA_tx"+str(i))
    rxn.name="Biomass_RNA"
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("DATP_p"+str(i)):-(1-GC)*0.79,cobra_model2.metabolites.get_by_id("aDATP_p"+str(i)):-(1-GC)*0.21,cobra_model2.metabolites.get_by_id("DGTP_p"+str(i)):-GC*0.50,cobra_model2.metabolites.get_by_id("aDGTP_p"+str(i)):-GC*0.10,cobra_model2.metabolites.get_by_id("bDGTP_p"+str(i)):-GC*0.40,cobra_model2.metabolites.get_by_id("DUTP_p"+str(i)):-(1-GC)*0.81,cobra_model2.metabolites.get_by_id("aDUTP_p"+str(i)):-(1-GC)*0.19,cobra_model2.metabolites.get_by_id("DCTP_p"+str(i)):-GC*0.79,cobra_model2.metabolites.get_by_id("aDCTP_p"+str(i)):-GC*0.21})
    met = Metabolite("RNA_b"+str(i))
    met.name="RNA_b"+str(i)
    met.compartment = "b"+str(i)
    rxn.add_metabolites({met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])

#### Step 2A: Setting up the initial phase

In [13]:
#The metabolites is added to initial temporal phase as boundary reactions entering the sysytem
#the concentrations of the metabolites converted to reaction bounds

scale = 1
unit_time = DPA[1]-DPA[0]

for k in Conc.keys():
    if k=="AA" or k=="Protein" or k=="NA":
        continue
    rxn = Reaction("Initial_"+str(k)+"_tx")
    rxn.name="Initial_"+str(k)
    if k=="STARCH":
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_p1"):1})
    else:
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_c1"):1})
    rxn.lower_bound = (float(Conc[k][0]))/scale
    rxn.upper_bound = (float(Conc[k][0]))/scale
    rxn.notes["influx_stage"]=1
    cobra_model2.add_reactions([rxn])

#### Step 2B: Setting up a biomass equation for the final phase using experimental data

In [14]:
#creatd final fruit biomass reaction based on experimental data (cite)

AA_b = ["GLN","ASN","SER","GLY","THR","L_ALPHA_ALANINE","4_AMINO_BUTYRATE","VAL","ILE","PHE","LEU","LYS","ARG","L_ASPARTATE","GLT","HIS","MET","PRO","TRP","TYR","CYS"]

for m in AA_b:
    met = Metabolite(m+"_b10")
    met.compartment = "b10"
    met.name = cobra_model2.metabolites.get_by_id(m+"_c10").name
    met.formula = cobra_model2.metabolites.get_by_id(m+"_c10").formula
    met.notes = cobra_model2.metabolites.get_by_id(m+"_c10").notes
    met.charge = cobra_model2.metabolites.get_by_id(m+"_c10").charge
    rxn = Reaction("Biomass_"+m+"_c_tx10")
    rxn.name="Biomass "+m+" _c accumulation"
    cobra_model2.add_reactions([rxn])
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_c10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000

    rxn = Reaction("Biomass_"+m+"_v_tx10")
    rxn.name="Biomass "+m+" _v accumulation"
    cobra_model2.add_reactions([rxn])
    if m == "HIS":
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id("b"+m+"_v10"):-1,met:1})
    else:
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_v10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000


OA = ["CIT","MAL","FUM"]
for m in OA:
    met = Metabolite(m+"_b10")
    met.compartment = "b10"
    met.name = cobra_model2.metabolites.get_by_id(m+"_c10").name
    met.formula = cobra_model2.metabolites.get_by_id(m+"_c10").formula
    met.notes = cobra_model2.metabolites.get_by_id(m+"_c10").notes
    met.charge = cobra_model2.metabolites.get_by_id(m+"_c10").charge
    rxn = Reaction("Biomass_"+m+"_c_tx10")
    rxn.name="Biomass "+m+" _c accumulation"
    cobra_model2.add_reactions([rxn])
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_c10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000

    rxn = Reaction("Biomass_"+m+"_v_tx10")
    rxn.name="Biomass "+m+" _v accumulation"
    cobra_model2.add_reactions([rxn])
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_v10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000


Sugars = ["GLC","FRU","SUCROSE"]
for m in Sugars:
    met = Metabolite(m+"_b10")
    met.compartment = "b10"
    met.name = cobra_model2.metabolites.get_by_id(m+"_c10").name
    met.formula = cobra_model2.metabolites.get_by_id(m+"_c10").formula
    met.notes = cobra_model2.metabolites.get_by_id(m+"_c10").notes
    met.charge = cobra_model2.metabolites.get_by_id(m+"_c10").charge
    rxn = Reaction("Biomass_"+m+"_c_tx10")
    rxn.name="Biomass "+m+" _c accumulation"
    cobra_model2.add_reactions([rxn])
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_c10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000

    rxn = Reaction("Biomass_"+m+"_v_tx10")
    rxn.name="Biomass "+m+" _v accumulation"
    cobra_model2.add_reactions([rxn])
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id(m+"_v10"):-1,met:1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    #######
    #rxn = Reaction("HYPO_biomass"+m)
    #rxn.add_metabolites({met:-1})
    #rxn.lower_bound = 0
    #rxn.upper_bound = 10
    #cobra_model2.add_reaction(rxn)




rxn = Reaction("Final_Biomass_tx")
rxn.name = "Final Fruit Biomass"
for k in Conc.keys():
    if k=="AA":
        for i in AA_b:
            rxn.add_metabolites({cobra_model2.metabolites.get_by_id(i+"_b10"):-float(Conc.get(k)[10])/scale})
    elif k=="Protein":
        continue
    elif k=="NA":
        continue
    else:
        if k=="STARCH":
            rxn.add_metabolites({cobra_model2.metabolites.get_by_id("Starch_b10"):-float(Conc.get(k)[10])/scale})
        elif k=="CELLULOSE":
            continue
        elif k=="PALMITATE":
            continue
        #elif k=="FUM":
        #    continue
        #elif k=="Pi":
        #    continue
        else:
            rxn.add_metabolites({cobra_model2.metabolites.get_by_id(k+"_b10"):-float(Conc.get(k)[10])/scale})

rxn.lower_bound = 0
rxn.upper_bound = 1000
cobra_model2.add_reactions([rxn])
rxn.objective_coefficient=1

for i in range(1,11):
    rxn = Reaction("Protein_biomass_demand_tx"+str(i))
    rxn.name = "Protein in Biomass "+str(i)
    k="Protein"
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("PROTEIN_b"+str(i)):-1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])

rxn = Reaction("Final_DNARNA_tx")
rxn.name = "Final NA in Biomass"
k="NA"
rxn.add_metabolites({cobra_model2.metabolites.get_by_id("RNA_b10"):-1,cobra_model2.metabolites.get_by_id("DNA_b10"):-1})
rxn.lower_bound = 0#float(Conc.get(k)[10])/scale
rxn.upper_bound = 0#float(Conc.get(k)[10])/scale
cobra_model2.add_reactions([rxn])


for i in range(10,11):
    rxn = Reaction("NITRATE_biomass"+str(i))
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("NITRATE_v"+str(i)):-1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])
    rxn = Reaction("MAL_biomass"+str(i))
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("MAL_v"+str(i)):-1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])
    rxn = Reaction("CIT_biomass"+str(i))
    rxn.add_metabolites({cobra_model2.metabolites.get_by_id("CIT_v"+str(i)):-1})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])
    for met in ["CAII","KI","MGII","NITRATE"]:
        rxn = Reaction(met+"_biomass_c"+str(i))
        rxn.add_metabolites({cobra_model2.metabolites.get_by_id(met+"_c"+str(i)):-1})
        rxn.lower_bound = 0
        rxn.upper_bound = 1000
        cobra_model2.add_reactions([rxn])


In [15]:
backup4 = cobra_model2.copy()

#### Since some amino acid catabolism pathways are not available in PlantCoreMetabolism 2.0, these amino acids were removed from the phloem uptake reaction

In [16]:
cobra_model2=backup4.copy()
#removed LEU and TRP from phloem because degradation pathway is incomplete and their fraction in
# phloem is extremely minor
# checking configuration using standerd FBA
for Met in ["LEU_c","TRP_c","VAL_c","ILE_c","PHE_c","4_AMINO_BUTYRATE_c","TYR_c","GLN_c"]:
    for i in range(1,11):
        met = cobra_model2.metabolites.get_by_id(Met+str(i))
        coeff = cobra_model2.reactions.get_by_id("Phloem_uptake_tx"+str(i)).metabolites.get(met)
        cobra_model2.reactions.get_by_id("Phloem_uptake_tx"+str(i)).add_metabolites({met:-1*coeff})

sol = cobra_model2.optimize()
sol.fluxes.get("Final_Biomass_tx")

np.float64(233.6561454259265)

Note that biomass accumulation rate is much higher than 1 suggesting there are not enough constriants on the system

# Part III: Set GrOE-FBA constraints to restrict osmolyte accumulation based on cell size



#### Step 3A: Adding GrOE-FBA structural constraints
##### Constraint cellulose demand based on the increasing pericarp volume

In [17]:
#Constraining cellulose demand flux based on increasing cell volume based on the data (cite)
#using the method and the function "celluloseDemandFlux" by (cite)

from studyFunctions import celluloseDemandFlux

# cobra_model2 = chargedFruit.copy()
print("Constraining cellulose demand flux...")


for i in range(1,11):
    rxn = Reaction("CELLULOSE_accumulation"+str(i))
    met = cobra_model2.metabolites.get_by_id("CELLULOSE_c"+str(i))
    rxn.add_metabolites({met:-1})
    rxn.lower_bound = celluloseDemandFlux(DPA[i],Ncells = 25*(10**6),unit_time=unit_time)
    rxn.upper_bound = celluloseDemandFlux(DPA[i],Ncells = 25*(10**6),unit_time=unit_time)
    cobra_model2.add_reactions([rxn])


Constraining cellulose demand flux...


##### Constraining phospholipid demand based on increasing pericarp volume

In [18]:
#Constraining palmitate demand flux based on increasing cell volume
#using the method and the function "phospholipidDemandFlux" by (cite)

from studyFunctions import phospholipidDemandFlux

print("Constraining phospholipid demand flux...")

for i in range(1,11):
    rxn = Reaction("phospholipid_accumulation"+str(i))
    met = cobra_model2.metabolites.get_by_id("L_1_PHOSPHATIDYL_ETHANOLAMINE_r"+str(i))
    rxn.add_metabolites({met:-0.273})
    met = cobra_model2.metabolites.get_by_id("PHOSPHATIDYL_CHOLINE_r"+str(i))
    rxn.add_metabolites({met:-0.353})
    met = cobra_model2.metabolites.get_by_id("L_PHOSPHATIDATE_p"+str(i))
    rxn.add_metabolites({met:-0.374})
    rxn.lower_bound = round(float(phospholipidDemandFlux(DPA[i],Ncells = 25*(10**6),unit_time=unit_time,scaling_factor=100.0/8.512820512820287)),5)
    rxn.upper_bound = round(float(phospholipidDemandFlux(DPA[i],Ncells = 25*(10**6),unit_time=unit_time,scaling_factor=100.0/8.512820512820287)),5)
    cobra_model2.add_reactions([rxn])

Constraining phospholipid demand flux...


In [19]:
#checking configuration
# cobra_model2 = chargedFruit.copy()
sol2 = cobra_model2.optimize()
sol2.fluxes.get("Final_Biomass_tx")

np.float64(233.65614542592633)

Note that biomass accumulation rate is still higher than 1, suggesting the system remains without enough constrains

##### Constraining protein demand based on pericarp volume

In [20]:
#Constraining protein demand flux based on increasing cell volume based on the data from (cite)
ProtConc=21458.1747597         #Biais data
#using the method and the function "estimateProteinDemandFlux" by (cite)

from studyFunctions import estimateProteinDemandFlux

print("Constraining protein demand flux...")
#cobra_model2 = temp_model.copy()
for i in range(1,11):
    rxn = cobra_model2.reactions.get_by_id("Protein_biomass_demand_tx"+str(i))
    print(estimateProteinDemandFlux(DPA[i],ProtConc=21458.1747597,unit_time=unit_time,Ncell = 25*(10**6)))
    temp_A = estimateProteinDemandFlux(DPA[i],ProtConc=21458.1747597,unit_time=unit_time,Ncell = 25*(10**6))
    rxn.lower_bound = round(temp_A,5)
    rxn.upper_bound = round(temp_A,5)

Constraining protein demand flux...
0.011281663377268358
0.017438573247776937
0.025997390503538213
0.030016941081953685
0.023765050046377884
0.013162547312008901
0.005670019340183796
0.0020806469990553023
0.0006689134501285943
0.0001768353519916721


In [21]:
#checking configuration
sol = cobra_model2.optimize()
sol.fluxes.get("Final_Biomass_tx")

np.float64(2.648434839663789)

Note that biomass accumulation is greater than 1 but considerably lower than earlier (~233) and while the model is closer to the appropriate level of constrains, more may be required

In [22]:
#checkpoint : create backup

backup2 =  cobra_model2.copy()

#### Step 3B: Setting up osmotic constraints for GrOE-FBA

In [23]:
#adding osmotic constraint

cobra_model2 = backup2.copy()


vacMets=["SUCROSE_v","MAL_v","NITRATE_v","AMMONIUM_v","CIT_v","GLN_v","ASN_v","SER_v","GLY_v","THR_v","L_ALPHA_ALANINE_v","4_AMINO_BUTYRATE_v","VAL_v","ILE_v","PHE_v","LEU_v","LYS_v","ARG_v","L_ASPARTATE_v","GLT_v","bHIS_v","MET_v","PRO_v","TRP_v","TYR_v","CYS_v","GLC_v","FRU_v","FUM_v","MGII_v","KI_v","CAII_v"]


#Add water potential constraint on the whole cell
for i in range(1,10):
    met1_v = Metabolite("VO_"+str(i))
    met1_v.name = "vacuolar osmolarity pseudo metabolite"
    met1_c = Metabolite("CO_"+str(i))
    met1_c.name = "cytosolic osmolarity pseudo metabolite"
    met2_c = Metabolite("CC_c"+str(i))
    met2_c.name = "cytosolic_charge_constraint"
    met2_c.compartment="c"+str(i)
    met2_v = Metabolite("CC_v"+str(i))
    met2_v.name = "vacuolar_charge_constraint"
    met2_v.compartment="v"+str(i)
    for metID in vacMets:
        met = cobra_model2.metabolites.get_by_id(metID+str(i))
        rxn = cobra_model2.reactions.get_by_id(metID+"_Transfer"+str(i)+str(i+1))
        charge = 0
        for Reac in rxn.reactants:
            if Reac.id.__contains__("PROTON"):
                continue
            charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        rxn.add_metabolites({met1_v:1,met2_v:charge})
    for rxn in cobra_model2.reactions.query("_c_Transfer"+str(i)+str(i+1)):
        charge = 0
        for Reac in rxn.reactants:
            if Reac.id.__contains__("PROTON"):
                continue
            charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        rxn.add_metabolites({met1_c:1,met2_c:charge})




#Set constraints for final fruit biomass
met1_v = Metabolite("VO_10")
met1_v.name = "vacuolar osmolarity pseudo metabolite"
met1_c = Metabolite("CO_10")
met1_c.name = "cytosolic osmolarity pseudo metabolite"
met2_c = Metabolite("CC_c10")
met2_c.name = "cytosolic_charge_constraint"
met2_c.compartment="c10"
met2_v = Metabolite("CC_v10")
met2_v.name = "vacuolar_charge_constraint"
met2_v.compartment="v10"

VO_10 = 0
CO_10 = 0
CC_c10 = 0
CC_v10 = 0

AA_b = ["GLN","ASN","SER","GLY","THR","L_ALPHA_ALANINE","4_AMINO_BUTYRATE","VAL","ILE","PHE","LEU","LYS","ARG","L_ASPARTATE","GLT","HIS","MET","PRO","TRP","TYR","CYS"]

for m in AA_b:
    met = cobra_model2.metabolites.get_by_id(m+"_c10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_c_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_c:stoich,met2_c:charge})
    met = cobra_model2.metabolites.get_by_id(m+"_v10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_v_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_v:stoich,met2_v:charge})


OA = ["CIT","MAL"]

for m in OA:
    met = cobra_model2.metabolites.get_by_id(m+"_c10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_c_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_c:stoich,met2_c:charge})
    met = cobra_model2.metabolites.get_by_id(m+"_v10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_v_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_v:stoich,met2_v:charge})

Sugars = ["GLC","FRU","SUCROSE"]

for m in Sugars:
    met = cobra_model2.metabolites.get_by_id(m+"_c10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_c_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_c:stoich,met2_c:charge})
    met = cobra_model2.metabolites.get_by_id(m+"_v10")
    rxn = cobra_model2.reactions.get_by_id("Biomass_"+m+"_v_tx10")
    charge = 0
    stoich = 0
    for Reac in rxn.reactants:
        if Reac.id.__contains__("PROTON"):
            continue
        charge = charge + (rxn.metabolites.get(Reac)*int(Reac.charge)*-1)
        stoich = stoich + (rxn.metabolites.get(Reac)*-1)
    rxn.add_metabolites({met1_v:stoich,met2_v:charge})


#
rxn = cobra_model2.reactions.get_by_id("Ca_biomass10")
rxn.add_metabolites({met1_v:1,met2_v:2})
rxn = cobra_model2.reactions.get_by_id("Mg_biomass10")
rxn.add_metabolites({met1_v:1,met2_v:2})
rxn = cobra_model2.reactions.get_by_id("K_biomass10")
rxn.add_metabolites({met1_v:1,met2_v:1})
rxn = cobra_model2.reactions.get_by_id("NITRATE_biomass10")
rxn.add_metabolites({met1_v:1,met2_v:-1})
rxn = cobra_model2.reactions.get_by_id("CAII_biomass_c10")
rxn.add_metabolites({met1_c:1,met2_c:2})
rxn = cobra_model2.reactions.get_by_id("MGII_biomass_c10")
rxn.add_metabolites({met1_c:1,met2_c:2})
rxn = cobra_model2.reactions.get_by_id("KI_biomass_c10")
rxn.add_metabolites({met1_c:1,met2_c:1})
rxn = cobra_model2.reactions.get_by_id("NITRATE_biomass_c10")
rxn.add_metabolites({met1_c:1,met2_c:-1})


import math
#Set total(met1_v) = (volume_of_vacuole/volume_of_cytosol) * total(met1_c)
for i in range(1,11):
    Vv = 0.853*(1-(math.e**((-2293-(DPA[i]*24*60))/10633)))
    Vc = (0.933 - Vv)/1.13
    #print(Vv/Vc)
    met = Metabolite("WCO_"+str(i))
    met.name = "Whole cell osmolarity psuedo metabolite"
    met1_v = cobra_model2.metabolites.get_by_id("VO_"+str(i))
    met1_c = cobra_model2.metabolites.get_by_id("CO_"+str(i))
    rxn =Reaction("IntercellularWaterPotentialConstraint"+str(i))
    rxn.add_metabolites({met1_c:-1,met1_v:-1*(Vc/Vv),met:1+(Vc/Vv)})
    rxn.lower_bound = 0
    rxn.upper_bound = 1000
    cobra_model2.add_reactions([rxn])


#Set sum(van't_Hoff_factor*number_of_moles) = cell_volume*Osmolarity
import math
C_cell = 275           #Almeida and Huber 1999; units = mOsmol/kg ~ mmol/L
C_cell = 275000        #units = mmol/m3
for i in range(1,11):
    met = cobra_model2.metabolites.get_by_id("WCO_"+str(i))
    V_pericarp = estimateVpericarp(DPA[i],hollow=False)
    rxn = Reaction("WCOsetter_tx"+str(i))
    rxn.name = "WCO_setter"
    rxn.add_metabolites({met:-1})
    rxn.lower_bound = round(V_pericarp*C_cell,3)
    rxn.upper_bound = round(V_pericarp*C_cell,3)
    cobra_model2.add_reactions([rxn])



In [24]:
sol = cobra_model2.optimize()
sol.fluxes.get("Final_Biomass_tx")

np.float64(1.0316915653448162)

Note that the biomass accumulation rate is approximately 1 suggesting that the system may be adequately constrained

# Part IV:Set non growth associated maintenance in the model

In [25]:
#NGAM based on maintenance respiration in Walker and Thornley 1997
#the demand set to 3:1 for ATP hydrolysis and NADPH oxidase
#ATPase flux set to 26.2

mR ={387:0.052,931:0.021,1591:0.015,2402:0.007}
for i in mR.keys():
    print("==========")
    print("mass ="+str(i))
    print("relative r="+str(mR[i]))
    print("mg/day ="+str(mR[i]*i))
    print("mmol/day ="+str(mR[i]*i/12))
    print("mmol/day ="+str(mR[i]*i*unit_time/12))

mass =387
relative r=0.052
mg/day =20.124
mmol/day =1.6769999999999998
mmol/day =8.5527
mass =931
relative r=0.021
mg/day =19.551000000000002
mmol/day =1.62925
mmol/day =8.309175
mass =1591
relative r=0.015
mg/day =23.865
mmol/day =1.9887499999999998
mmol/day =10.142624999999999
mass =2402
relative r=0.007
mg/day =16.814
mmol/day =1.4011666666666667
mmol/day =7.145949999999999


In [26]:
meanRes = (8.553+8.308+10.144+7.145)/4
print(meanRes)

8.537500000000001


In [27]:
ATPase = 26.2

for i in range(1,11):
    met=Metabolite("ATPNAPDH_maintenance_constraint_"+str(i))
    cobra_model2.reactions.get_by_id("ATPase_tx"+str(i)).add_metabolites({met:-1})
    cobra_model2.reactions.get_by_id("NADPHoxc_tx"+str(i)).add_metabolites({met:3})
    cobra_model2.reactions.get_by_id("NADPHoxp_tx"+str(i)).add_metabolites({met:3})
    cobra_model2.reactions.get_by_id("NADPHoxm_tx"+str(i)).add_metabolites({met:3})
    cobra_model2.reactions.get_by_id("ATPase_tx"+str(i)).lower_bound = ATPase
    cobra_model2.reactions.get_by_id("ATPase_tx"+str(i)).upper_bound = ATPase

In [28]:
#creating backup and checking configuration
backup4 = cobra_model2.copy()
sol =backup4.optimize()
print(sol.fluxes.get("Final_Biomass_tx"))

1.0316915653448202


# Part V: Introduce additional metabolite influx and efflux constraints based on available data

In [29]:
#constraining phloem uptake rate

cobra_model2 = backup4.copy()
C=0
for met in cobra_model2.reactions.get_by_id("Phloem_uptake_tx1").metabolites.keys():
    if met.formula == "" or met.formula == "NA" or not "C" in met.formula:
        continue
    C=C+(int(met.formula.split("H")[0].split("C")[1])*cobra_model2.reactions.get_by_id("Phloem_uptake_tx1").metabolites.get(met))

print("Total C in 1 mol of phloem = "+str(abs(C)))
print("-------------")

for i in range(1,11):
    Ccont = estimateCcontent(DPA[i])
    cobra_model2.reactions.get_by_id("Phloem_uptake_tx"+str(i)).upper_bound = estimatePhloemUptakeConstraint(Ccont)*24*(unit_time/abs(C*12))
    print("Cuptake rate = "+str(estimatePhloemUptakeConstraint(Ccont)*24*(unit_time/abs(C*12))))


Total C in 1 mol of phloem = 9.8517340867
-------------
Cuptake rate = 10.324678854928035
Cuptake rate = 9.857056994937958
Cuptake rate = 7.105057458316657
Cuptake rate = 4.091525576332845
Cuptake rate = 3.1757082904689846
Cuptake rate = 2.9719412716083764
Cuptake rate = 2.921891273238981
Cuptake rate = 2.9076536488828175
Cuptake rate = 2.9030444117861642
Cuptake rate = 2.901383124541366


# Part VI: Use weighted parsimonious FBA to study metabolic fluxes

In [30]:
# run simulations - pFBA

weightings = dict()
for rxn in cobra_model2.reactions:
    weightings[rxn.id]=1

for k in Conc.keys():
    for i in range(1,10):
        if k != "STARCH":
            met = Metabolite(k+str(i))
            rxn = cobra_model2.reactions.get_by_id(k+"_c_Transfer"+str(i)+str(i+1))
            rxn.add_metabolites({met:1})
            rxn = cobra_model2.reactions.get_by_id(k+"_v_Transfer"+str(i)+str(i+1))
            rxn.add_metabolites({met:1})
            rxn=Reaction(k+"_signal"+str(i))
            rxn.add_metabolites({met:-1})
            rxn.lower_bound = 0
            rxn.upper_bound = 1000
            cobra_model2.add_reactions([rxn])
            weightings[rxn.id]=0

solution2 = pfba_Weighted(cobra_model2,weightings)


In [31]:
print(solution2.fluxes.get("Final_Biomass_tx"))

1.0316915653448242
